# **SILOAM PLAYSTORE**

In [ ]:
from google_play_scraper import Sort, reviews_all
import pandas as pd
from datetime import datetime

# ID aplikasi
app_id = 'com.siloam.android'

# ===== FILTER TANGGAL =====
cutoff_start = datetime(2019, 5, 1)
cutoff_end = datetime(2025, 11, 30, 23, 59, 59)
# ==========================

print("Mengambil ulasan dari Google Play Store...")

reviews = reviews_all(
    app_id,
    lang='id',
    country='id',
    sort=Sort.NEWEST
)

# Konversi ke DataFrame
df_playstore = pd.DataFrame(reviews)

# =========================
# PROSES & FILTER TANGGAL
# =========================
df_playstore["at"] = pd.to_datetime(df_playstore["at"])

# Filter sesuai rentang tanggal
df_playstore = df_playstore[
    (df_playstore["at"] >= cutoff_start) &
    (df_playstore["at"] <= cutoff_end)
].copy()

# Urutkan terbaru
df_playstore = df_playstore.sort_values("at", ascending=False).reset_index(drop=True)

print(f"Total ulasan setelah filter: {len(df_playstore)}")

if not df_playstore.empty:
    print(f"Rentang tanggal: {df_playstore['at'].min()} s/d {df_playstore['at'].max()}")

# =========================
# SIMPAN FILE
# =========================
df_playstore.to_csv('MySiloam_reviews_playstore_filtered.csv', index=False, encoding='utf-8-sig')
df_playstore.to_excel('MySiloam_reviews_playstore_filtered.xlsx', index=False)

print("File disimpan:")
print(" - MySiloam_reviews_playstore_filtered.csv")
print(" - MySiloam_reviews_playstore_filtered.xlsx")

# Preview
df_playstore[["userName", "score", "content", "at"]].head()

Mengambil ulasan dari Google Play Store...
Total ulasan setelah filter: 2376
Rentang tanggal: 2019-05-24 06:32:33 s/d 2025-11-29 07:07:02
File disimpan:
 - MySiloam_reviews_playstore_filtered.csv
 - MySiloam_reviews_playstore_filtered.xlsx


,userName,score,content,at
0,Pengguna Google,3,setiap mau masuk ke beranda selalu loadingg,2025-11-29 07:07:02
1,Pengguna Google,5,Aplikasi RS terbaik...,2025-11-27 07:24:22
2,Pengguna Google,5,"aplikasi lengkap, fitur andalan dan sangat inf...",2025-11-27 06:22:26
3,Pengguna Google,5,ok,2025-11-26 00:13:26
4,Pengguna Google,5,good,2025-11-25 05:34:25


# ***SCARPPING APPS STORE***

In [ ]:
import requests
import pandas as pd
from datetime import datetime

In [ ]:
# App ID untuk "MySiloam" di App Store
app_id = "1456325611"
country = "id"
# CATATAN: Apple RSS feed hanya menyediakan 10 halaman terakhir.
max_pages = 10

all_reviews = []

print(f"Konfigurasi siap untuk App ID: {app_id} (Maks {max_pages} halaman)")

Konfigurasi siap untuk App ID: 1456325611 (Maks 10 halaman)


In [ ]:
import requests
import pandas as pd
from datetime import datetime

app_id = "1456325611"
country = "id"
max_pages = 10
all_reviews = []

cutoff_end = datetime(2025, 11, 30, 23, 59, 59)

print(f"⏳ Mulai mengambil ulasan MySiloam dari App Store...\n")

stop = False

for page in range(1, max_pages + 1):
    if stop:
        break

    url = f"https://itunes.apple.com/{country}/rss/customerreviews/page={page}/id={app_id}/json"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"❌ Gagal halaman {page}: {e}")
        break

    if "feed" not in data or "entry" not in data["feed"]:
        print(f"🚫 Tidak ada data di halaman {page}, berhenti.")
        break

    entries = data["feed"]["entry"][1:]
    print(f"📦 Halaman {page}: {len(entries)} ulasan")

    for e in entries:
        tanggal_raw = e.get("updated", {}).get("label", "")

        try:
            tanggal = datetime.fromisoformat(tanggal_raw.replace("Z", "+00:00")).replace(tzinfo=None)
        except:
            tanggal = None

        # 👉 STOP kalau sudah lebih dari cutoff
        if tanggal and tanggal > cutoff_end:
            continue

        nama = e.get("author", {}).get("name", {}).get("label", "N/A")
        rating = int(e.get("im:rating", {}).get("label", "0"))
        ulasan = e.get("content", {}).get("label", "")

        all_reviews.append({
            "nama": nama,
            "rating": rating,
            "ulasan": ulasan,
            "tanggal": tanggal
        })

print(f"\n🎯 Total ulasan terkumpul: {len(all_reviews)}")

# =========================
# DATAFRAME (lebih ringan)
# =========================
df = pd.DataFrame(all_reviews)

df["tanggal"] = pd.to_datetime(df["tanggal"])
df = df.sort_values("tanggal", ascending=False).reset_index(drop=True)

print(f"📅 Rentang: {df['tanggal'].min()} s/d {df['tanggal'].max()}")

# Simpan
df.to_excel("mysiloam_appstore_optimized.xlsx", index=False)

print("\n📁 File disimpan:")
print("   - mysiloam_appstore_optimized.xlsx")

df[["nama", "rating", "ulasan", "tanggal"]].head()

⏳ Mulai mengambil ulasan MySiloam dari App Store...

📦 Halaman 1: 49 ulasan
📦 Halaman 2: 49 ulasan
📦 Halaman 3: 49 ulasan
📦 Halaman 4: 49 ulasan
📦 Halaman 5: 49 ulasan
📦 Halaman 6: 49 ulasan
📦 Halaman 7: 1 ulasan
🚫 Tidak ada data di halaman 8, berhenti.

🎯 Total ulasan terkumpul: 281
📅 Rentang: 2019-06-18 18:57:08 s/d 2025-11-28 19:03:22

📁 File disimpan:
   - mysiloam_appstore_optimized.xlsx


,nama,rating,ulasan,tanggal
0,Alfiishere,5,Very easy to use and helpful!,2025-11-28 19:03:22
1,Kusni73,5,Awalnya terasa susah ternyata lebih mudah..,2025-11-21 02:51:40
2,philipsumolang,5,fitur self check out sangat membantu,2025-11-06 06:23:03
3,Fauzannnnnnnnnnnnn,1,saya pilih chat doctor buat konsultasi terkait...,2025-11-01 16:35:58
4,Eli kistina,5,Siloam paling terbaik rumah sakit di balikpapa...,2025-09-20 01:58:18
